## **Expiry-Day Liquidity Concentration in Indian Index Derivatives**

### **Notebook: 05: Robustness Checks**

---

**Scope:** Placebo cutover dates, leave-one-out exclusion of each treated expiry, and an event-study leads/lags specification, probing whether the primary result is a genuine discontinuity rather than an artifact of a longer-running trend or a handful of unusual observations.

---

### Pipeline sequence

1. **01_data_collection.ipynb** -- index OHLCV, expiry calendar, holidays, F&O Bhavcopy archive
2. **02_feature_engineering_and_fo_construction.ipynb** -- technical/volatility features; lot-size-invariant turnover and open interest
3. **03_weekly_options_discontinuation_identification.ipynb** -- dating the Nov 2024 weekly-contract removal directly from exchange records
4. **04_primary_diff_in_diff_analysis.ipynb** -- the headline difference-in-differences result (Bank Nifty vs. Nifty 50)
5. **05_robustness_checks.ipynb** -- placebo cutover dates, leave-one-out, event-study leads/lags
6. **06_expiry_volume_event_study.ipynb** -- descriptive expiry-day volume/OI patterns by regime
7. **07_walk_forward_predictability.ipynb** -- secondary analysis: leakage-free return-predictability test (null result)
8. **08_interrupted_time_series.ipynb** -- secondary analysis: settlement-weekday effect on returns/volatility (null result)
9. **09_figures_for_paper.ipynb** -- regenerates all manuscript figures

**Current location:** **05_robustness_checks.ipynb** <- *you are here*


### Cached robustness results

The full robustness battery (placebo dates, leave-one-out, and the event-study regression) is run by `scripts.diff_in_diff_banknifty_vs_nifty` and `scripts.event_study_leads_lags`. Rebuild with:

```bash
python -m scripts.diff_in_diff_banknifty_vs_nifty
python -m scripts.event_study_leads_lags
```

This notebook loads the saved results.

In [1]:
import sys
from pathlib import Path

project_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_dir) not in sys.path:
    sys.path.append(str(project_dir))

import pandas as pd
import json
from src import config

pd.set_option('display.max_columns', 20)
print('Project root:', project_dir)

Project root: E:\BN_Structure_Reforms\banknifty-expiry-volatility-ml


In [2]:
with open(config.RESULTS_DIR / 'diff_in_diff_banknifty_vs_nifty.json') as f:
    did = json.load(f)

print('=== Placebo-date test (Total_Value) ===')
for label, r in did['placebo_date_test_Total_Value'].items():
    print(f"{label}: coef={r['coef']:.4f}, p={r['p_value']:.4f}")

=== Placebo-date test (Total_Value) ===
placebo cutover 2021-06-01 (deep in Weekly-Options era, no real break): coef=0.6203, p=0.0095
placebo cutover 2022-06-01 (deep in Weekly-Options era, no real break): coef=0.8338, p=0.0011
placebo cutover 2023-06-01 (deep in Weekly-Options era, no real break): coef=1.0888, p=0.0000
TRUE cutover 2024-11-14: coef=1.3418, p=0.0000


In [3]:
loo = did['leave_one_out_Total_Value']
print('=== Leave-one-out (23 Bank Nifty Monthly-Only expiry days) ===')
print(f"Full-sample coefficient: {loo['full_sample_coef']:.4f}")
print(f"Range across exclusions: [{loo['loo_coef_min']:.4f}, {loo['loo_coef_max']:.4f}]")
print(f"Mean: {loo['loo_coef_mean']:.4f}, SD: {loo['loo_coef_std']:.4f}")

=== Leave-one-out (23 Bank Nifty Monthly-Only expiry days) ===
Full-sample coefficient: 1.3418
Range across exclusions: [1.2170, 1.4786]
Mean: 1.3418, SD: 0.0463


In [4]:
with open(config.RESULTS_DIR / 'event_study_leads_lags.json') as f:
    event_study = json.load(f)

pd.DataFrame(event_study['Total_Value']).sort_values('period_half_years_from_event')

,period_half_years_from_event,coef,p_value,ci_lower,ci_upper
0,-10,0.269111,2.499895e-01,-0.189390,0.727611
8,-2,0.562198,1.075738e-02,0.130167,0.994230
7,-3,0.140802,1.569069e-01,-0.054152,0.335756
6,-4,0.290311,2.863918e-02,0.030306,0.550315
5,-5,0.236962,4.510923e-04,0.104577,0.369347
4,-6,0.165574,1.142286e-01,-0.039887,0.371036
3,-7,0.265925,4.429481e-03,0.082779,0.449071
2,-8,0.300034,1.317987e-03,0.116955,0.483113
1,-9,0.290452,8.036071e-03,0.075676,0.505228
9,0,1.198680,6.263506e-02,-0.063219,2.460579
